In [4]:
import time
from typing import TypedDict, Literal

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.constants import START, END
from langgraph.graph import StateGraph, MessagesState
from langgraph.prebuilt import ToolRuntime, ToolNode
from langgraph.runtime import Runtime


# 1 定义状态
class OverAllState(TypedDict):
    initial_state: str
    node_a_output: str
    node_b_output: str


# 2、定义节点
# runtime.stream_writer实现流式输出内容
def node_a(state: OverAllState, runtime: Runtime) -> OverAllState:
    stream_writer = runtime.stream_writer
    stream_writer("节点A正在执行...")
    time.sleep(2)
    return {
        "node_a_output": "节点A的输出"
    }


def node_b(state: OverAllState, runtime: Runtime) -> OverAllState:
    stream_writer = runtime.stream_writer
    stream_writer("节点B正在执行...")
    time.sleep(2)
    return {
        "node_b_output": "节点B的输出"
    }


# 3、构建图
builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_a", node_a)
builder.add_node("node_b", node_b)
builder.add_edge(START, "node_a")
builder.add_edge("node_a", "node_b")
builder.add_edge("node_b", END)

graph = builder.compile()
for chunk in graph.stream(
        {"initial_state": "初始状态"},
        # 如果想要节点输出流式内容，必须 添加 stream_mode=["custom"]
        stream_mode=["custom"]
):
    print(chunk)

('custom', '节点A正在执行...')
('custom', '节点B正在执行...')


In [5]:
from langchain.tools import tool

load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)


# 工具调用
@tool(parse_docstring=True)
def get_weather(city: str, runtime: ToolRuntime) -> str:
    """
    根据城市查询当日的天气

    Args:
        city: 城市名称
    """
    stream_writer = runtime.stream_writer
    stream_writer(f"正在查询 {city} 今天的天气...")
    return f"{city} 今天天气不错"


tools = [get_weather]
model_with_tools = model.bind_tools(tools=tools)


# 大模型节点
def llm_node(state: MessagesState, runtime: Runtime) -> MessagesState:
    messages = state["messages"]
    response = model_with_tools.invoke(messages)

    stream_writer = runtime.stream_writer
    stream_writer("正在执行 llm_node...")

    return {
        "messages": [response]
    }


# 路由节点
def router(state: MessagesState) -> Literal["tool_node", END]:
    last_msg = state["messages"][-1]
    if last_msg.tool_calls:
        return "tool_node"
    return END


builder = StateGraph(state_schema=MessagesState)
builder.add_node("llm_node", llm_node)
builder.add_node("tool_node", ToolNode(tools=tools))
builder.add_edge(START, "llm_node")
builder.add_conditional_edges("llm_node", router, path_map=["tool_node", END])
builder.add_edge("tool_node", "llm_node")

graph = builder.compile()

for chunk in graph.stream(
        {"messages": [HumanMessage("今天北京天气怎么样")]},
        # 用户自定义流式输出 custom
        stream_mode=["custom"]
): print(chunk)

('custom', '正在执行 llm_node...')
('custom', '正在查询 北京 今天的天气...')
('custom', '正在执行 llm_node...')
